# Pima Indians Diabetes — Exploratory Data Analysis


A walkthrough of the UCI **Pima Indians Diabetes Dataset** (768 rows, 8 features, binary `Outcome`). The goal is *understanding*, not modeling: what's genuinely missing vs. recorded as zero, how the two groups differ per feature, and which features correlate most with the label.

All analysis lives in `eda.py` so the notebook, `run.py`, and `tests/` share one implementation.


## 1. Load & first look


In [ ]:
import numpy as np
import pandas as pd
import eda

raw = eda.load()
print(raw.shape)
raw.head()

(768, 9)


 Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin  BMI  DiabetesPedigreeFunction  Age  Outcome
           6      148             72             35        0 33.6                     0.627   50        1
           1       85             66             29        0 26.6                     0.351   31        0
           8      183             64              0        0 23.3                     0.672   32        1
           1       89             66             23       94 28.1                     0.167   21        0
           0      137             40             35      168 43.1                     2.288   33        1

The columns map straight onto the UCI schema. One thing to watch: several `0`s below (glucose, blood pressure, skin thickness, insulin, BMI) are *not* physically meaningful — they're "not measured" placeholders. We'll handle that next.


## 2. Zero-as-missing coercion


A glucose of 0 is impossible; the export just wrote `0` for missing values. `coerce_zeros()` converts those to NaN and reports how many, so every downstream stat reflects real observations.


In [ ]:
df, coerced = eda.coerce_zeros(raw)
print('zeros coerced to NaN per column:')
for col, n in sorted(coerced.items()):
    if n:
        print(f'  {col}: {n}')

BMI: 11
BloodPressure: 35
Glucose: 5
Insulin: 374
SkinThickness: 227


In [ ]:
eda.missing_report(df)

                  column  missing  pct_missing severity
                 Insulin      374        48.70   medium
           SkinThickness      227        29.56   medium
           BloodPressure       35         4.56      low
                     BMI       11         1.43      low
                 Glucose        5         0.65      low
             Pregnancies        0         0.00      low
DiabetesPedigreeFunction        0         0.00      low
                     Age        0         0.00      low
                 Outcome        0         0.00      low

`Insulin` (~49%) and `SkinThickness` (~30%) are heavily incomplete; the rest are clean. Any model that naively keeps those zeros would be trained partly on artifacts — worth remembering.


## 3. Outcome distribution


In [ ]:
oc = eda.outcome_counts(raw)
print(f"{oc['outcome_1']} positive / {oc['outcome_0']} negative "
      f"({oc['positive_rate']:.1%} positive)")

268 positive / 500 negative (34.9% positive)


The class is mildly imbalanced (~35% positive). Not severe, but a baseline model that always predicts negative would already be ~65% accurate — so any metric here should be read with that in mind.


## 4. Feature means by outcome


`feature_means_by_outcome()` shows each feature's mean within each group plus the delta (positive minus negative). Big deltas flag candidate predictors.


In [ ]:
eda.feature_means_by_outcome(df)

                          outcome_0_mean  outcome_1_mean   delta
feature                                                         
Pregnancies                        3.298           4.866   1.568
Glucose                          110.644         142.320  31.676
BloodPressure                     70.877          75.321   4.444
SkinThickness                     27.235          33.000   5.765
Insulin                          130.288         206.846  76.558
BMI                               30.860          35.407   4.547
DiabetesPedigreeFunction           0.430           0.550   0.121
Age                               31.190          37.067   5.877

The largest separations are **Glucose** (+31.7) and **Insulin** (+76.6) — both central to diabetes pathophysiology, so the signal is physiologically sensible. `Pregnancies`, `Age`, and `BMI` also lean positive.


## 5. Correlations


In [ ]:
eda.correlation_matrix(df)

                          Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin    BMI  DiabetesPedigreeFunction    Age  Outcome
Pregnancies                     1.000    0.128          0.214          0.100    0.082  0.022                    -0.034  0.544    0.222
Glucose                         0.128    1.000          0.223          0.228    0.581  0.233                     0.137  0.267    0.495
BloodPressure                   0.214    0.223          1.000          0.227    0.098  0.289                    -0.003  0.330    0.171
SkinThickness                   0.100    0.228          0.227          1.000    0.185  0.648                     0.115  0.167    0.259
Insulin                         0.082    0.581          0.098          0.185    1.000  0.228                     0.130  0.220    0.303
BMI                             0.022    0.233          0.289          0.648    0.228  1.000                     0.155  0.026    0.314
DiabetesPedigreeFunction       -0.034    0.137         

In [ ]:
eda.top_correlates_with_outcome(df, k=5)

Glucose          0.494650
BMI              0.313680
Insulin          0.303454
SkinThickness    0.259491
Age              0.238356

**Glucose** is the standout single correlate with `Outcome` (|r| ≈ 0.495); BMI, Insulin, and SkinThickness follow. Note the feature-feature block too: `Insulin`↔`Glucose` (0.581) and `BMI`↔`SkinThickness` (0.648) are meaningfully correlated, so a multivariate model will see some redundancy there.


## 6. Takeaways


- **Missingness is real**: ~49% of Insulin and ~30% of SkinThickness are missing *after* zero-coercion — impute or drop, don't trust the zeros.
- **Glucose is the primary signal** (correlation 0.495); a single-feature model on it would already be informative.
- **Mild imbalance** (35% positive) means accuracy alone is a weak metric.
- **Feature redundancy** (Insulin↔Glucose, BMI↔SkinThickness) suggests a regularized or tree-based model rather than plain logistic regression on all features.
- Next step (out of scope for pure EDA): a train/validation split with a simple baseline and a proper metric like ROC-AUC.
